In [ ]:
# imports
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

In [7]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
deepseek_url = "https://api.deepseek.com"


gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)

In [8]:
# open source models
# qwen2.5-coder (Alibaba)
# deepseek-coder-v2 (DeepSeek)
# gpt-oss:20b (OpenAI)
# qwen/qwen3-coder-30b-a3b-instruct (Alibaba)
# openai/gpt-oss-120b (OpenAI)
# model:clientlibrary
models = ["qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b" ]
clients = {"openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": ollama}



In [9]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustc.EXE',
  'version': 'rustc 1.95.0 (59807616e 2026-04-14)',
  'host_triple': 'x86_64-pc-windows-gnu',
  'release': '1.95.0',
  'commit_hash': '59807616e1fa2540724bfbac14d7976d7e4a3860'},
 'cargo': {'path': 'C:\\Users\\HP\\.cargo\\bin\\cargo.EXE',
  'version': 'cargo 1.95.0 (f2d3ce0bd 2026-03-21)'},
 'rustup': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-x86_64-pc-windows-gnu (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-pc-windows-gnu (active, default)',
   'stable-x86_64-pc-windows-msvc'],
  'targets_installed': ['x86_64-pc-windows-gnu']},
 'rust_analyzer': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\HP\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\HP\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\\Users\\HP\\.cargo\\bi

In [10]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = groq.chat.completions.create(model=models[4], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))


**You already have a Rust toolchain installed**  

The system report shows:

* `rustc 1.95.0` at `C:\Users\HP\.cargo\bin\rustc.EXE`  
* `cargo 1.95.0` at `C:\Users\HP\.cargo\bin\cargo.EXE`  
* `rustup` with the `stable‑x86_64-pc-windows-gnu` toolchain active  

So you do **not** need to install anything else to compile a single‑file program.

---

## Fastest‑runtime compilation on Windows (GNU toolchain)

To get the absolute best runtime performance you want a *release‑quality* build with:

| Flag | Why it helps |
|------|--------------|
| `-C opt-level=3` | Highest optimization level. |
| `-C target-cpu=native` | Generates code tuned for your exact CPU (i5‑4030U). |
| `-C lto=fat` | Whole‑program Link‑Time Optimization. |
| `-C codegen-units=1` | Forces a single code‑generation unit → better inlining across the whole crate. |
| `-C debuginfo=0` | No debug info → smaller binary, slightly faster load. |
| `-C panic=abort` (optional) | Removes unwinding code; makes the binary a bit smaller/faster. |
| `-C strip=debuginfo` (optional) | Strips any remaining debug symbols from the final exe. |

All of these can be passed directly to `rustc`.

### Python snippet

Below is a minimal, **self‑contained** Python example that compiles `main.rs` with the flags above, then runs the produced binary and returns its stdout.

```python
import subprocess
import pathlib

# ----------------------------------------------------------------------
# 1️⃣  Compile the Rust source (fastest possible runtime)
# ----------------------------------------------------------------------
rustc = r"C:\Users\HP\.cargo\bin\rustc.EXE"
src   = "main.rs"                     # path to your source file
out   = "main.exe"                    # name of the produced binary

compile_command = [
    rustc,
    src,
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "debuginfo=0",
    "-C", "panic=abort",            # optional – removes unwind tables
    "-C", "strip=debuginfo",        # optional – strip any leftover symbols
    "-o", out
]

# Run the compilation
compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True,
)
print("Compilation finished.")
print("stdout:", compile_result.stdout)
print("stderr:", compile_result.stderr)

# ----------------------------------------------------------------------
# 2️⃣  Execute the produced binary
# ----------------------------------------------------------------------
run_command = [str(pathlib.Path(out).resolve())]

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True,
)

# The function can now return the program's output:
print("Program output:")
print(run_result.stdout)
```

#### What the commands do

* **`compile_command`** – Calls `rustc.exe` with the aggressive optimization flags listed above and writes the executable to `main.exe`.
* **`run_command`** – Executes the generated `main.exe`.  
  Using the absolute path (`Path(...).resolve()`) avoids any “file not found” issues when the current working directory changes.

---

## If you ever need to (re‑)install Rust

> *You don’t need this right now, but it’s handy to keep for future machines.*

```powershell
# 1️⃣ Install rustup (the Rust toolchain manager)
# Download the installer and run it
iwr https://win.rustup.rs -UseBasicParsing | iex

# 2️⃣ Add the GNU toolchain (already present, but shown for completeness)
rustup toolchain install stable-x86_64-pc-windows-gnu

# 3️⃣ Make it the default
rustup default stable-x86_64-pc-windows-gnu
```

---

### TL;DR – What you should copy‑paste into Python

```python
compile_command = [
    r"C:\Users\HP\.cargo\bin\rustc.EXE",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "debuginfo=0",
    "-C", "panic=abort",
    "-C", "strip=debuginfo",
    "-o", "main.exe",
]

run_command = ["main.exe"]
```

These commands give you the **fastest possible runtime performance** on your Windows‑10, x86_64‑pc‑windows‑gnu platform, while still being simple to invoke from Python. Happy coding!

In [ ]:
compile_command = [
    "rustc",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=yes",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "main.rs",
    "-o", "main.exe",
]

# Fix: Use a dot-backslash or just "main.exe"
run_command = [r".\main.exe"]

## And now, on with the main task

In [12]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{extension} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [13]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [23]:
def write_output(code):
    with open(f"main.{extension}", "w",encoding="utf-8") as f:
        f.write(code)

In [ ]:
def port(model, python):
    client = clients[model]
    
    kwargs = {
        "model": model, 
        "messages": messages_for(python),
        "temperature": 0.0, 
    }
    
    if 'gpt' in model and 'groq' not in str(client.base_url):
        kwargs["reasoning_effort"] = "high"
        
    print(f"Sending request to {model}...")
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    
    if not reply:
        print(f"❌ Error: {model} returned an empty payload string block.")
        return ""
        
    print(f"Response received from {model} ({len(reply)} chars). Parsing...")

    # --- FIXED: Robust Multi-stage Language Independent Extraction ---
    if "```cpp" in reply:
        reply = reply.split("```cpp")[1].split("```")[0].strip()
    elif "```rust" in reply:
        reply = reply.split("```rust")[1].split("```")[0].strip()
    elif "```" in reply:
        reply = reply.split("```")[1].split("```")[0].strip()
    else:
        # If no markdown code block backticks were used at all, strip out accidental stragglers safely
        reply = reply.replace('```cpp','').replace('```rust','').replace('```','').strip()
        
    write_output(reply)
    print(f"✅ Code successfully cleaned and written!")
    return reply


In [16]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# Use the commands from GPT 5
import subprocess
import os

def compile_and_run(code):
    write_output(code)
    try:
        print("Compiling source code...")
        # Added shell=True to handle compiler binary mapping smoothly
        subprocess.run(compile_command, check=True, text=True, capture_output=True, shell=True)
        print("Compilation successful! Running binary...")
        
        # Added shell=True here to let Windows resolve the '.\' relative path safely
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True, shell=True)
        return run_result.stdout
        
    except subprocess.CalledProcessError as e:
        # Catches compiler or code syntax crashes
        return f"❌ Compilation/Execution failed with exit code {e.returncode}:\n{e.stderr if e.stderr else e.stdout}"
        
    except FileNotFoundError:
        # Catches cases where the executable path name is mismatched or missing
        current_dir_files = os.listdir(".")
        return (
            f"❌ Windows Error: Could not find the executable specified in run_command: {run_command}\n"
            f"Please verify your Cell 8 configuration matches the output filename.\n"
            f"Current directory contents: {current_dir_files[:10]}..."
        )

In [18]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [25]:
rust_code = port("openai/gpt-oss-120b",python_hard)

Sending request to openai/gpt-oss-120b...
Response received from openai/gpt-oss-120b (1483 chars). Parsing...
✅ Code successfully cleaned and written!


In [26]:
compile_and_run(rust_code)

Compiling source code...
Compilation successful! Running binary...


'Total Maximum Subarray Sum (20 runs): 10980\nExecution Time: 0.001213 seconds\n'

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


C:\Users\HP\AppData\Local\Temp\ipykernel_3148\3949588770.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
ui.close()

Closing server running on port: 7860


In [ ]:
import subprocess

result = subprocess.run(
    ["rustc", "--version"],
    capture_output=True,
    text=True
)

print(result.stdout)

rustc 1.95.0 (59807616e 2026-04-14)

